# Phase 2 — Pharmaprojects check

Three tests, in the order of how much new information they carry:

1. **PAV versus non-PAV enrichment.** Never tested on Pharmaprojects. The genuinely new result.
2. **The frozen PAV + 2–5 therapeutic-area definition.** OR and RS, no refitting.
3. **Non-linear pleiotropy.** Expected to fail on power. The achieved power is computed *before* the
   test is run, so a negative result is a pre-stated limitation rather than a surprise.

## What this is not

Pharmaprojects is independent *curation* of the same pharmacological reality, not an independent
dataset. Approved drugs are approved drugs: the launched set overlaps ChEMBL's phase-4 set heavily
(quantified below, and in `01_build_pair_tables.ipynb`: 447 of 911 launched T–I pairs are also ChEMBL
phase 4, 49%). Nothing here may be described as replication in independent data. What it does test is
whether the result survives a different curation of clinical outcome, different indication mapping
(MeSH → EFO/MONDO rather than ChEMBL's own EFO), and a different phase definition.

## Data provenance and its self-validation

`ti_pairs_pharmaprojects_master-r1.parquet` is built in `01_build_pair_tables.ipynb` from
`minikel_etal_processed_data_v2.csv`, the processed Pharmaprojects table from
`chapters/05-other-drug-indication-data/01-process-minikel_etal_data.ipynb` — gene symbols mapped to
Ensembl ids, MeSH indications mapped through the disease index, oncology removed, `ccatnum >= 2`
(phase I or beyond), `outcome = (ccatnum == 5)` for launched. Our propagated genetic-support and PAV
columns are joined onto it.

Before using it: regressing launch on Pharmaprojects' **own** genetic-support flag gives
OR = 2.32 [1.94, 2.78], P = 1.8e-20 — the ~2× that Nelson 2015 and Minikel 2024 report. The table is
therefore joined correctly and carries the signal it should. That check is repeated here.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2

from or10_stats import or_rs, support_mask, window_label

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

path_to_intermediate_data_folder = "../../../data/intermediate_files/"
pp = pd.read_parquet(path_to_intermediate_data_folder + "ti_pairs_pharmaprojects_master-r1.parquet")
chembl = pd.read_parquet(path_to_intermediate_data_folder + "ti_pairs_chembl_master-r1.parquet")

for df in (pp, chembl):
    df["ta"] = df["uniqueTherapeuticAreas"].fillna(0).astype(float)
    df["support_all"] = support_mask(df).astype(int)
    df["support_pav"] = support_mask(df, pav=True).astype(int)

PUBLISHED_OR = 10.288962
PUBLISHED_RS = 4.843708
CHEMBL_BASELINE_OR = 3.618578

print("Pharmaprojects pairs:", len(pp), "| targets:", pp["targetId"].nunique())
print("launched pairs:", int(pp["approved"].sum()))
print(
    pp.groupby("ccatnum")
    .size()
    .rename("pairs")
    .to_frame()
    .assign(phase=lambda d: ["I", "II", "III", "launched"])
    .to_string()
)
print()
print(
    "with our all-GWAS support:",
    int(pp["support_all"].sum()),
    "| with PAV support:",
    int(pp["support_pav"].sum()),
    "| with PAV + 2-5 TA support:",
    int(support_mask(pp, pav=True, ta_min=2, ta_max=5).sum()),
)

Pharmaprojects pairs: 7390 | targets: 1164
launched pairs: 913
         pairs     phase
ccatnum                 
2         2274         I
3         3303        II
4          900       III
5          913  launched

with our all-GWAS support: 469 | with PAV support: 137 | with PAV + 2-5 TA support: 60


## Self-validation and overlap

In [2]:
own = smf.logit("approved ~ geneticSupport_old", data=pp).fit(disp=False)
or_own = float(np.exp(own.params["geneticSupport_old"]))
ci_own = np.exp(own.conf_int().loc["geneticSupport_old"].values)
print(
    f"Pharmaprojects own genetic-support flag: OR = {or_own:.3f} [{ci_own[0]:.3f}, {ci_own[1]:.3f}], "
    f"p = {own.pvalues['geneticSupport_old']:.3g}"
)
assert 1.9 < or_own < 2.8, or_own  # the ~2x reported by Nelson 2015 / Minikel 2024

pp_launched = set(map(tuple, pp.loc[pp["approved"] == 1, ["targetId", "diseaseId"]].values))
chembl_ph4 = set(map(tuple, chembl.loc[chembl["approved"] == 1, ["targetId", "diseaseId"]].values))
shared = pp_launched & chembl_ph4
print(
    f"launched T-I pairs shared with ChEMBL phase 4: {len(shared)} of {len(pp_launched)} "
    f"({100 * len(shared) / len(pp_launched):.1f}%)"
)

Pharmaprojects own genetic-support flag: OR = 2.323 [1.944, 2.777], p = 1.84e-20
launched T-I pairs shared with ChEMBL phase 4: 447 of 911 (49.1%)


## Test 1 — PAV versus non-PAV enrichment

Two readings of the same question, both reported:

- **Fisher, separately for each stratum.** PAV support versus no support, and support-without-PAV
  versus no support. Directly comparable to the published ChEMBL ORs of 6.0 and 3.1.
- **Logistic regression with a three-level evidence factor**, the coefficients compared by a t-test
  on the contrast. This is the specification the published stratified comparisons used, and it is the
  one that gives a P value for the *difference*.

In [3]:
def stratified_pav(df, label):
    """PAV versus non-PAV genetic support, Fisher per stratum plus the contrast test."""
    no_support = df["support_all"] == 0
    pav = df["support_pav"] == 1
    nonpav = (df["support_all"] == 1) & (df["support_pav"] == 0)

    rows = []
    for name, mask in [
        ("support with PAV", pav),
        ("support without PAV", nonpav),
        ("any support", df["support_all"] == 1),
    ]:
        # compare this stratum against the no-support pairs only, so the strata are not each other's control
        sub = df[mask | no_support]
        res = or_rs(mask.loc[sub.index], sub["approved"])
        rows.append({"dataset": label, "stratum": name, **res})
    table = pd.DataFrame(rows)

    e = np.where(df["support_all"] == 0, 0, np.where(df["support_pav"] == 1, 2, 1))
    data = pd.DataFrame({"outcome": df["approved"].to_numpy(), "E": e})
    fit = smf.logit("outcome ~ C(E)", data=data).fit(disp=False)
    contrast = np.zeros(len(fit.params))
    contrast[1], contrast[2] = 1, -1
    contrast_p = float(np.ravel(fit.t_test(contrast).pvalue)[0])
    logistic = {
        "dataset": label,
        "or_nonpav": float(np.exp(fit.params.iloc[1])),
        "or_pav": float(np.exp(fit.params.iloc[2])),
        "p_nonpav": float(fit.pvalues.iloc[1]),
        "p_pav": float(fit.pvalues.iloc[2]),
        "p_difference": contrast_p,
        "n_nonpav": int((e == 1).sum()),
        "n_pav": int((e == 2).sum()),
        "n_approved_nonpav": int(data.loc[e == 1, "outcome"].sum()),
        "n_approved_pav": int(data.loc[e == 2, "outcome"].sum()),
    }
    return table, logistic


pp_strata, pp_logistic = stratified_pav(pp, "Pharmaprojects")
chembl_strata, chembl_logistic = stratified_pav(chembl, "ChEMBL")

print(
    pd.concat([chembl_strata, pp_strata])[
        [
            "dataset",
            "stratum",
            "odds_ratio",
            "ci_low",
            "ci_high",
            "relative_success",
            "n_support",
            "yes_evid-high_clinphase",
        ]
    ]
    .round(3)
    .to_string(index=False)
)
print()
pav_comparison = pd.DataFrame([chembl_logistic, pp_logistic])
print(pav_comparison.round(4).to_string(index=False))

       dataset             stratum  odds_ratio  ci_low  ci_high  relative_success  n_support  yes_evid-high_clinphase
        ChEMBL    support with PAV       6.048   4.426    8.265             3.791        161                       72
        ChEMBL support without PAV       3.092   2.579    3.708             2.480        581                      170
        ChEMBL         any support       3.619   3.094    4.233             2.765        742                      242
Pharmaprojects    support with PAV       2.338   1.570    3.482             2.016        137                       33
Pharmaprojects support without PAV       1.400   1.034    1.894             1.336        332                       53
Pharmaprojects         any support       1.655   1.295    2.114             1.535        469                       86

       dataset  or_nonpav  or_pav  p_nonpav  p_pav  p_difference  n_nonpav  n_pav  n_approved_nonpav  n_approved_pav
        ChEMBL     3.0924  6.0483    0.0000    0.0      

In [4]:
print("Published ChEMBL values for this comparison: OR 6.0 (PAV) versus 3.1 (no PAV), p = 0.0002")
print(
    f"Reproduced here on ChEMBL:                   OR {chembl_logistic['or_pav']:.2f} versus "
    f"{chembl_logistic['or_nonpav']:.2f}, p = {chembl_logistic['p_difference']:.2g}"
)
print(
    f"Pharmaprojects (new):                        OR {pp_logistic['or_pav']:.2f} versus "
    f"{pp_logistic['or_nonpav']:.2f}, p = {pp_logistic['p_difference']:.2g}"
    f"  [{pp_logistic['n_approved_pav']} launched of {pp_logistic['n_pav']} PAV-supported pairs]"
)

Published ChEMBL values for this comparison: OR 6.0 (PAV) versus 3.1 (no PAV), p = 0.0002
Reproduced here on ChEMBL:                   OR 6.05 versus 3.09, p = 0.00024
Pharmaprojects (new):                        OR 2.34 versus 1.40, p = 0.04  [33 launched of 137 PAV-supported pairs]


## Test 2 — the frozen PAV + 2–5 therapeutic-area definition

No refitting, no window search: the published definition applied verbatim to the Pharmaprojects
pairs. The ChEMBL row is shown alongside so the comparison is like-for-like, and the counts make the
precision visible — this is a small number of launched pairs.

In [5]:
frozen_rows = []
for label, df in [("ChEMBL", chembl), ("Pharmaprojects", pp)]:
    for name, kwargs in [
        ("all GWAS support", {}),
        ("PAV, any TA", {"pav": True}),
        ("PAV + 2-5 TA (published)", {"pav": True, "ta_min": 2, "ta_max": 5}),
        ("any support + 2-5 TA", {"ta_min": 2, "ta_max": 5}),
    ]:
        frozen_rows.append({"dataset": label, "definition": name, **or_rs(support_mask(df, **kwargs), df["approved"])})

frozen = pd.DataFrame(frozen_rows)
frozen[
    [
        "dataset",
        "definition",
        "odds_ratio",
        "ci_low",
        "ci_high",
        "relative_success",
        "ci_rs_low",
        "ci_rs_high",
        "p_value",
        "n_support",
        "yes_evid-high_clinphase",
    ]
].round(3)

,dataset,definition,odds_ratio,ci_low,ci_high,relative_success,ci_rs_low,ci_rs_high,p_value,n_support,yes_evid-high_clinphase
0,ChEMBL,all GWAS support,3.619,3.094,4.233,2.765,2.484,3.077,0.000,742,242
1,ChEMBL,"PAV, any TA",5.893,4.313,8.053,3.705,3.114,4.409,0.000,161,72
2,ChEMBL,PAV + 2-5 TA (published),10.289,6.708,15.782,4.844,4.051,5.791,0.000,87,51
3,ChEMBL,any support + 2-5 TA,3.948,3.205,4.863,2.919,2.545,3.347,0.000,398,139
4,Pharmaprojects,all GWAS support,1.655,1.295,2.114,1.535,1.255,1.877,0.000,469,86
5,Pharmaprojects,"PAV, any TA",2.298,1.544,3.421,1.985,1.465,2.690,0.000,137,33
6,Pharmaprojects,PAV + 2-5 TA (published),4.498,2.660,7.605,3.157,2.277,4.377,0.000,60,23
7,Pharmaprojects,any support + 2-5 TA,1.845,1.300,2.618,1.673,1.264,2.214,0.001,202,41


In [6]:
pp_strict = frozen[(frozen["dataset"] == "Pharmaprojects") & (frozen["definition"] == "PAV + 2-5 TA (published)")].iloc[
    0
]
pp_all = frozen[(frozen["dataset"] == "Pharmaprojects") & (frozen["definition"] == "all GWAS support")].iloc[0]

print(
    f"Pharmaprojects, frozen published definition: OR {pp_strict['odds_ratio']:.2f} "
    f"[{pp_strict['ci_low']:.2f}, {pp_strict['ci_high']:.2f}], RS {pp_strict['relative_success']:.2f}, "
    f"p = {pp_strict['p_value']:.2g}"
)
print(
    f"  carried by {int(pp_strict['yes_evid-high_clinphase'])} launched pairs of "
    f"{int(pp_strict['n_support'])} supported"
)
print(
    f"Pharmaprojects, all-GWAS support:            OR {pp_all['odds_ratio']:.2f} "
    f"[{pp_all['ci_low']:.2f}, {pp_all['ci_high']:.2f}], RS {pp_all['relative_success']:.2f}"
)
print(
    f"  ratio strict / all-GWAS: {pp_strict['odds_ratio'] / pp_all['odds_ratio']:.2f} "
    f"(ChEMBL: {PUBLISHED_OR / CHEMBL_BASELINE_OR:.2f})"
)

Pharmaprojects, frozen published definition: OR 4.50 [2.66, 7.60], RS 3.16, p = 2.5e-07
  carried by 23 launched pairs of 60 supported
Pharmaprojects, all-GWAS support:            OR 1.65 [1.30, 2.11], RS 1.53
  ratio strict / all-GWAS: 2.72 (ChEMBL: 2.84)


### Therapeutic-area window profile on Pharmaprojects

Purely descriptive, and thin: reported with the launched-pair count against every row so no cell can
be read as a result on its own.

In [7]:
profile_rows = []
for lo, hi in [(1, 4), (1, 5), (2, 4), (2, 5), (2, 6), (3, 6), (2, None), (None, None)]:
    profile_rows.append(
        {
            "window": window_label(lo, hi),
            **or_rs(support_mask(pp, pav=True, ta_min=lo, ta_max=hi), pp["approved"]),
        }
    )
pp_profile = pd.DataFrame(profile_rows)
print(
    pp_profile[
        ["window", "odds_ratio", "ci_low", "ci_high", "relative_success", "n_support", "yes_evid-high_clinphase"]
    ]
    .round(3)
    .to_string(index=False)
)

window  odds_ratio  ci_low  ci_high  relative_success  n_support  yes_evid-high_clinphase
   1-4       3.802   2.172    6.658             2.834         55                       19
   1-5       3.947   2.389    6.522             2.907         68                       24
   2-4       4.472   2.473    8.085             3.142         47                       18
   2-5       4.498   2.660    7.605             3.157         60                       23
   2-6       4.717   2.865    7.767             3.253         66                       26
   3-6       3.898   2.253    6.746             2.881         57                       20
   >=2       2.389   1.592    3.585             2.044        129                       32
   all       2.298   1.544    3.421             1.985        137                       33


## Test 3 — non-linear pleiotropy

### Achieved power, computed before the test

The ChEMBL model is taken as truth: fit
`approved ~ geneticSupport + log(TA + 1) + log(TA + 1)²` on ChEMBL, transplant the slopes onto the
Pharmaprojects design matrix (its own therapeutic-area distribution and support pattern), recalibrate
the intercept so the simulated launch rate matches the observed one, simulate outcomes and run the
same likelihood-ratio test. Power is the fraction of simulations detecting the quadratic term at
P < 0.05.

**Two effect sizes, because one of them would be dishonest on its own.** Every genetic-support effect
is weaker in Pharmaprojects than in ChEMBL — all-GWAS support gives OR 1.65 there versus 3.62 here.
Assuming ChEMBL-strength coefficients therefore overstates the power available:

- **full strength** — ChEMBL slopes unchanged. Answers "if the effect were as strong here as there".
- **attenuation-matched** — all slopes multiplied by
  `log(OR_all-GWAS, Pharmaprojects) / log(OR_all-GWAS, ChEMBL)`, so the simulated genetic-support
  effect matches the one Pharmaprojects actually shows. This is the fair power figure.

If power is low under the attenuation-matched scenario, a null result says nothing about whether the
non-linearity is real, and must be reported as such.

In [8]:
def fit_nonlinear(data):
    """Nested logistic fits for the quadratic log(TA + 1) term."""
    m0 = smf.logit("outcome ~ geneticSupport", data=data).fit(disp=False)
    m1 = smf.logit("outcome ~ geneticSupport + logta", data=data).fit(disp=False)
    m2 = smf.logit("outcome ~ geneticSupport + logta + logta2", data=data).fit(disp=False)
    return m0, m1, m2


def nonlinear_result(df, label):
    """Likelihood-ratio test for the quadratic term, on one dataset."""
    data = pd.DataFrame(
        {
            "outcome": df["approved"].to_numpy(),
            "geneticSupport": df["support_all"].to_numpy(),
            "logta": np.log(df["ta"].to_numpy() + 1),
        }
    )
    data["logta2"] = data["logta"] ** 2
    m0, m1, m2 = fit_nonlinear(data)
    lr_21 = 2 * (float(m2.llf) - float(m1.llf))
    lr_20 = 2 * (float(m2.llf) - float(m0.llf))
    b, a = float(m2.params["logta"]), float(m2.params["logta2"])
    return (
        {
            "dataset": label,
            "n": len(data),
            "n_outcome": int(data["outcome"].sum()),
            "lr_quadratic": lr_21,
            "p_quadratic": float(chi2.sf(lr_21, 1)),
            "lr_both_terms": lr_20,
            "p_both_terms": float(chi2.sf(lr_20, 2)),
            "coef_logta": b,
            "coef_logta2": a,
            "peak_ta": float(np.exp(-b / (2 * a)) - 1) if a != 0 else np.nan,
        },
        m2,
        data,
    )


chembl_nl, chembl_m2, _ = nonlinear_result(chembl, "ChEMBL")
print(pd.DataFrame([chembl_nl]).round(4).to_string(index=False))
assert np.isclose(chembl_nl["lr_quadratic"], 64.897, atol=0.05)

dataset     n  n_outcome  lr_quadratic  p_quadratic  lr_both_terms  p_both_terms  coef_logta  coef_logta2  peak_ta
 ChEMBL 37377       4564       64.8973          0.0        80.5394           0.0      0.5684      -0.2667    1.903


In [9]:
pp_design = pd.DataFrame(
    {
        "geneticSupport": pp["support_all"].to_numpy(),
        "logta": np.log(pp["ta"].to_numpy() + 1),
    }
)
pp_design["logta2"] = pp_design["logta"] ** 2

slopes = chembl_m2.params
observed_rate = float(pp["approved"].mean())
N_POWER_SIMS = 500

# how much weaker every effect is in Pharmaprojects, on the log-odds scale
pp_support_or = float(or_rs(support_mask(pp), pp["approved"])["odds_ratio"])
attenuation = np.log(pp_support_or) / np.log(CHEMBL_BASELINE_OR)
print(
    f"all-GWAS support: OR {CHEMBL_BASELINE_OR:.2f} in ChEMBL, {pp_support_or:.2f} in Pharmaprojects "
    f"-> attenuation factor {attenuation:.3f} on the log-odds scale"
)


def power_for(scale, label, seed=20260811):
    """Power of the quadratic LRT on the Pharmaprojects design, with slopes scaled by `scale`."""
    linear = (
        scale
        * (
            slopes["geneticSupport"] * pp_design["geneticSupport"]
            + slopes["logta"] * pp_design["logta"]
            + slopes["logta2"] * pp_design["logta2"]
        ).to_numpy()
    )

    lo_int, hi_int = -12.0, 4.0
    for _ in range(80):  # bisect the intercept onto the observed launch rate
        mid = (lo_int + hi_int) / 2
        if float(np.mean(1 / (1 + np.exp(-(mid + linear))))) < observed_rate:
            lo_int = mid
        else:
            hi_int = mid
    intercept = (lo_int + hi_int) / 2
    probs = 1 / (1 + np.exp(-(intercept + linear)))

    rng = np.random.default_rng(seed)
    p_values = np.empty(N_POWER_SIMS)
    for i in range(N_POWER_SIMS):
        sim = pp_design.copy()
        sim["outcome"] = (rng.random(len(sim)) < probs).astype(int)
        _, m1, m2 = fit_nonlinear(sim)
        p_values[i] = chi2.sf(2 * (float(m2.llf) - float(m1.llf)), 1)

    summary = {
        "scenario": label,
        "slope_scale": scale,
        "simulated_rate": float(probs.mean()),
        "observed_rate": observed_rate,
        "power_p05": float((p_values < 0.05).mean()),
        "power_p01": float((p_values < 0.01).mean()),
        "median_simulated_p": float(np.median(p_values)),
    }
    print(
        f"{label}: power {100 * summary['power_p05']:.1f}% at p<0.05, "
        f"{100 * summary['power_p01']:.1f}% at p<0.01, median simulated p = {summary['median_simulated_p']:.3g}"
    )
    return summary, p_values


full_power, p_full = power_for(1.0, "full ChEMBL strength")
att_power, p_att = power_for(attenuation, "attenuation-matched")
power_table = pd.DataFrame([full_power, att_power])
power_05 = att_power["power_p05"]  # the fair figure, used in the verdict below

all-GWAS support: OR 3.62 in ChEMBL, 1.65 in Pharmaprojects -> attenuation factor 0.392 on the log-odds scale


full ChEMBL strength: power 99.4% at p<0.05, 96.2% at p<0.01, median simulated p = 1.25e-05


attenuation-matched: power 43.0% at p<0.05, 19.8% at p<0.01, median simulated p = 0.0703


In [10]:
pp_nl, pp_m2, _ = nonlinear_result(pp, "Pharmaprojects")
nonlinearity = pd.DataFrame([chembl_nl, pp_nl])
nonlinearity["power_p05_full_strength"] = [np.nan, full_power["power_p05"]]
nonlinearity["power_p05_attenuation_matched"] = [np.nan, att_power["power_p05"]]
print(nonlinearity.round(4).to_string(index=False))
print()

pp_ci = pp_m2.conf_int().loc["logta2"].values
chembl_ci = chembl_m2.conf_int().loc["logta2"].values
print(
    f"quadratic coefficient: ChEMBL {chembl_nl['coef_logta2']:.4f} [{chembl_ci[0]:.4f}, {chembl_ci[1]:.4f}], "
    f"Pharmaprojects {pp_nl['coef_logta2']:.4f} [{pp_ci[0]:.4f}, {pp_ci[1]:.4f}]"
)
print(
    f"  the ChEMBL point estimate is "
    f"{'inside' if pp_ci[0] <= chembl_nl['coef_logta2'] <= pp_ci[1] else 'outside'} "
    f"the Pharmaprojects interval"
)
print()
verdict = "detected" if pp_nl["p_quadratic"] < 0.05 else "not detected"
print(f"quadratic term on Pharmaprojects: {verdict} (p = {pp_nl['p_quadratic']:.3g})")
print(f"  power to detect a full-strength ChEMBL effect:   {100 * full_power['power_p05']:.0f}%")
print(f"  power to detect an attenuation-matched effect:   {100 * att_power['power_p05']:.0f}%")
if pp_nl["p_quadratic"] >= 0.05:
    if power_05 < 0.8:
        print("=> a pre-stated power limitation, not evidence against the non-linearity")
    else:
        print("=> powered to detect it and did not: report as a failure to replicate the non-linearity,")
        print("   while noting the whole genetic-support effect is weaker in this resource")

       dataset     n  n_outcome  lr_quadratic  p_quadratic  lr_both_terms  p_both_terms  coef_logta  coef_logta2  peak_ta  power_p05_full_strength  power_p05_attenuation_matched
        ChEMBL 37377       4564       64.8973       0.0000        80.5394        0.0000      0.5684      -0.2667   1.9030                      NaN                            NaN
Pharmaprojects  7390        913        0.4661       0.4948         0.8622        0.6498      0.1160      -0.0424   2.9298                    0.994                           0.43

quadratic coefficient: ChEMBL -0.2667 [-0.3331, -0.2002], Pharmaprojects -0.0424 [-0.1643, 0.0795]
  the ChEMBL point estimate is outside the Pharmaprojects interval

quadratic term on Pharmaprojects: not detected (p = 0.495)
  power to detect a full-strength ChEMBL effect:   99%
  power to detect an attenuation-matched effect:   43%
=> a pre-stated power limitation, not evidence against the non-linearity


## Export

In [11]:
pd.concat([chembl_strata, pp_strata]).to_csv(
    path_to_intermediate_data_folder + "or10_phase2_pav_strata-r1.csv", index=False
)
pav_comparison.to_csv(path_to_intermediate_data_folder + "or10_phase2_pav_contrast-r1.csv", index=False)
frozen.to_csv(path_to_intermediate_data_folder + "or10_phase2_frozen_definition-r1.csv", index=False)
pp_profile.to_csv(path_to_intermediate_data_folder + "or10_phase2_window_profile-r1.csv", index=False)
nonlinearity.to_csv(path_to_intermediate_data_folder + "or10_phase2_nonlinearity-r1.csv", index=False)
power_table.to_csv(path_to_intermediate_data_folder + "or10_phase2_power-r1.csv", index=False)
pd.DataFrame({"p_full_strength": p_full, "p_attenuation_matched": p_att}).to_csv(
    path_to_intermediate_data_folder + "or10_phase2_power_simulation-r1.csv", index=False
)
print("exported 7 tables")

exported 7 tables
